In [1]:
from rag import LinkedInPostsRAG

In [6]:
from dotenv import load_dotenv
load_dotenv('/Users/shahules/Myprojects/notes/.envrc')

True

In [3]:
rag = LinkedInPostsRAG("/Users/shahules/Myprojects/notes/superme/data/shahules_posts.json")
await rag.initialize()

Loaded 122 LinkedIn posts
BM25 index initialized
Loaded existing document vectors from /Users/shahules/Myprojects/notes/superme/data/shahules_posts_vectors.npy


In [4]:
query = "When did you become a kaggle GM?"
response = await rag.answer_query(query, retrieval_method="bm25", top_k=2)


In [5]:
response

RagResponse(answer='The context does not provide information about becoming a Kaggle Grandmaster (GM), only a Kaggle Master.', reasoning='The only mention of Kaggle status in the provided posts is becoming a Kaggle Master, not GM. Typically, a Kaggle Grandmaster is a more advanced and distinguished status compared to a Master, and such an achievement would likely be explicitly mentioned if it occurred. Therefore, based on the given information, the user became a Kaggle Master but there is no indication of becoming a Kaggle Grandmaster.', sources=['https://www.linkedin.com/posts/shahules_kaggle-datasciences-jobplacement-activity-6595952946943496192-bitg?utm_source=share&utm_medium=member_desktop&rcm=ACoAACHEQ1kBau5gFVkfSgsSB2flft8HtbfWS74'])

In [6]:
query = "When did you become a kaggle GM?"
response = await rag.answer_query(query, retrieval_method="agentic", top_k=2)


In [7]:
response

RagResponse(answer='The exact date when they became a Kaggle Grandmaster is not specified in the provided posts, but they are listed as a Kaggle Grandmaster (Kernels) since September 2021.', reasoning='The profile information from Post 1 states that the individual has held the title of Kaggle Grandmaster (Kernels) starting from September 2021. However, without a specific date, it is not possible to determine the exact day they achieved this milestone. The other posts do not provide exact dates for becoming a Grandmaster, but Post 2 and Post 3, both from 5 years ago, show progression in their Kaggle achievements, which likely contributed to eventually attaining the Grandmaster status.', sources=['https://www.linkedin.com/in/shahules/'])

In [8]:
from ragas_annotator.project.core import Project


In [19]:
os.environ.get("NOTION_TOKEN")


'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'

In [7]:
import os
from ragas_annotator.project.core import Project

PROJECT_ID = '1a65d9bf94ff8061a82dd8dc31d69949'
NOTION_TOKEN = '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'

project = Project(
    name="Ragas Dashboard", 
    notion_api_key=NOTION_TOKEN, 
    notion_root_page_id=PROJECT_ID,
)


In [11]:
from ragas_annotator.model.notion_model import NotionModel
from ragas_annotator import nmt
import typing as t

class Dataset(NotionModel):
    id: str = nmt.ID()
    query: str = nmt.Title()
    user_id: str = nmt.Select()
    grading_notes: str = nmt.Text()

In [11]:
dataset = project.get_dataset(
    name="Test Shahul AI",
    model=Dataset,
)
dataset.load()

In [ ]:
class Experiment(Dataset):
    response: str = nmt.Text()
    correctness: str = nmt.Select()
    correctness_reason: str = nmt.Text()
    trace_url: str = nmt.Text()
    correctness_traces: str
    
exp = project.get_experiment(exp_name, Experiment)
exp.load()


In [13]:
from ragas_annotator.tracing.langfuse import *
async def get_current_trace():
    trace = await sync_trace(max_retries=10, delay=5)
    trace_url = trace.get_url()
    trace_url = add_query_param(trace_url, "trace_id", trace.trace.id)
    return trace_url

In [14]:
from langfuse.decorators import langfuse_context, observe
 
@observe()
async def answer_query(query: str, retrieval_method: str = "bm25", top_k: int = 2):
    response = await rag.answer_query(query, retrieval_method=retrieval_method, top_k=top_k)
    trace_url = langfuse_context.get_current_trace_url()
    return response, trace_url
 


response,trace_url = await answer_query(query, retrieval_method="bm25", top_k=2)


In [8]:
from ragas_annotator.llm import ragas_llm
from ragas_annotator.metric import DiscreteMetric
from openai import AsyncOpenAI

llm = ragas_llm(provider="openai",model="gpt-4o",client=AsyncOpenAI())

my_metric = DiscreteMetric(
    llm=llm,
    name='correctness',
    prompt="Given the Question: {query} \n Evaluate if given answer {response} \n based on the Grading notes\n: {grading_notes}.",
    values=["pass","fail"],
)

    
# test LLM as judge
result = my_metric.score(query="what is your response", response="this is my response",grading_notes="- response should not contains word response")
result

'fail'

In [16]:
from ragas_annotator.embedding import ragas_embedding

from openai import OpenAI
embedding = ragas_embedding(provider='openai',client=OpenAI(),model="text-embedding-3-small")
my_metric.train(project,experiment_names=['gradonfly-experiment-0'],embedding_model=embedding,model=Experiment,method={})


Processing examples: 100%|██████████| 11/11 [00:03<00:00,  3.43it/s]


In [17]:

@project.langfuse_experiment(Experiment, name_prefix="Workshop")
async def run_experiment(row: Dataset):
    response,trace_url = await answer_query(row.query, retrieval_method="agentic", top_k=5)
    result = await my_metric.ascore(
        query=row.query,
        response=response.answer,
        grading_notes=row.grading_notes,
    )
    experiment_view = Experiment(
        id=row.id,
        user_id=row.user_id,
        query=row.query,
        grading_notes=row.grading_notes,
        response=response.answer,
        trace_url=trace_url,
        correctness=result.result,
        correctness_reason=result.reason,
    )
    
    return experiment_view

In [18]:
await run_experiment.run_async(
    name="gradonfly-experiment-5",
    dataset=dataset
)

100%|██████████| 11/11 [00:11<00:00,  1.04s/it]


Experiment(name=gradonfly-experiment-5, model=Experiment)

In [23]:
def compare_experiments(project, experiment_names, metric):

    results = []
    for exp_name in experiment_names:
        exp = project.get_experiment(exp_name, Experiment)
        exp.load()
        values = [getattr(row, metric.name) for row in exp]
        results.append({exp_name:values})
    return results
        
   
   


In [24]:
results = compare_experiments(project,["gradonfly-experiment-5","gradonfly-experiment-4","gradonfly-experiment-3","gradonfly-experiment-1","gradonfly-experiment-0"],my_metric)

In [25]:
import plotly.graph_objects as go
import pandas as pd

# Process the data to get pass/fail percentages for each experiment
processed_data = []
for exp_obj in results:
    exp_name = list(exp_obj.keys())[0]
    results = exp_obj[exp_name]
    
    # Count passes and fails
    pass_count = results.count('pass')
    fail_count = results.count('fail')
    
    # Calculate percentages
    total_count = len(results)
    pass_percentage = (pass_count / total_count) * 100
    fail_percentage = (fail_count / total_count) * 100
    
    processed_data.append({
        'experiment': exp_name,
        'pass_percentage': pass_percentage,
        'fail_percentage': fail_percentage,
        'pass_count': pass_count,
        'fail_count': fail_count,
        'total': total_count
    })

# Sort data from experiment-3 to experiment-5
processed_data.sort(key=lambda x: int(x['experiment'].split('-')[-1]))

# Create arrays for Plotly input
experiments = [d['experiment'] for d in processed_data]
pass_percentages = [d['pass_percentage'] for d in processed_data]
fail_percentages = [d['fail_percentage'] for d in processed_data]

# Create the stacked bar chart
fig = go.Figure()

# Add pass trace
fig.add_trace(go.Bar(
    x=experiments,
    y=pass_percentages,
    name='Pass',
    marker_color='rgb(0, 153, 76)'
))

# Add fail trace
fig.add_trace(go.Bar(
    x=experiments,
    y=fail_percentages,
    name='Fail',
    marker_color='rgb(204, 51, 51)'
))

# Update layout
fig.update_layout(
    title='Experiment Results: Pass vs Fail Percentages',
    barmode='stack',
    xaxis=dict(title='Experiment Name'),
    yaxis=dict(title='Percentage (%)', range=[0, 100]),
    legend=dict(x=0.7, y=1),
    margin=dict(l=60, r=40, b=80, t=60),
    width=800, 
    height=500
)

# Add annotations for pass rates
for i, exp in enumerate(experiments):
    pass_rate = pass_percentages[i]
    pass_count = processed_data[i]['pass_count']
    total = processed_data[i]['total']
    
    fig.add_annotation(
        x=exp,
        y=pass_rate/2,  # Position in middle of pass section
        text=f"{pass_rate:.1f}%<br>({pass_count}/{total})",
        showarrow=False,
        font=dict(color="white", size=12)
    )
    
    # Add annotation for fail rates
    fail_rate = fail_percentages[i]
    fail_count = processed_data[i]['fail_count']
    
    fig.add_annotation(
        x=exp,
        y=pass_rate + fail_rate/2,  # Position in middle of fail section
        text=f"{fail_rate:.1f}%<br>({fail_count}/{total})",
        showarrow=False,
        font=dict(color="white", size=12)
    )

# Show the figure
fig.show()

# Optional: Save the figure
# fig.write_html("experiment_results.html")
# fig.write_image("experiment_results.png")

In [28]:
exp_x, exp_y = "gradonfly-experiment-0", "gradonfly-experiment-5"
exp_x_data = project.get_experiment(exp_x, Experiment)
exp_y_data = project.get_experiment(exp_y, Experiment)
exp_x_data.load()
exp_y_data.load()

# Compare experiments (assuming this is a function that exists)
project.compare_experiments(exp_x_data, exp_y_data)

Experiments have different models: <class '__main__.Experiment'> and <class '__main__.Experiment'>
Experiments have different models: <class '__main__.Experiment'> and <class '__main__.Experiment'>
Uploading to Notion: 100%|██████████| 11/11 [00:36<00:00,  3.33s/it]


'https://www.notion.so/1d15d9bf94ff81ed88cad27e6cd87c62'